Importações

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

import random

Seed Global

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


Carregamento do Dataset Anotado

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

caminho_arquivo = '/content/drive/MyDrive/Linguamática/merge_anotacoes.csv'
df = pd.read_csv(caminho_arquivo)

df = df[df["anotacao_final"].notna()].copy()
df["frase"] = df["frase"].astype(str).fillna("").str.strip()
df = df[df["frase"] != ""].reset_index(drop=True)

X = df["frase"].values
y = df["anotacao_final"].values

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

num_classes = len(label_encoder.classes_)
print("Classes:", label_encoder.classes_)
print("Número de classes:", num_classes)

Mounted at /content/drive
Classes: ['NÃO' 'SIM']
Número de classes: 2


In [ ]:
class_counts = np.bincount(y)
print("Contagem de classes:")
for i, count in enumerate(class_counts):
    class_name = label_encoder.inverse_transform([i])[0]
    print(f"  {class_name}: {count}")

Contagem de classes:
  NÃO: 7153
  SIM: 15013


Dataset Customizado com Tokenização

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": str(self.texts[idx]),
            "labels": int(self.labels[idx])
        }

collate function com dynamic padding

In [ ]:
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def collate_fn(batch):
    texts = [b["text"] for b in batch]
    labels = torch.tensor([b["labels"] for b in batch], dtype=torch.long)

    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )
    enc["labels"] = labels
    return enc

Função de Treino

In [ ]:
import torch
import torch.nn as nn

def train_model(model, data_loader, optimizer, device, loss_fn=None, scheduler=None, use_amp=True, max_grad_norm=1.0):
    model.train()
    total_loss = 0.0

    amp_enabled = (use_amp and device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

    for batch in data_loader:
        optimizer.zero_grad(set_to_none=True)

        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch["labels"]

        with torch.amp.autocast("cuda", enabled=amp_enabled):
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )
            logits = outputs.logits

            if loss_fn is None:
                loss = nn.functional.cross_entropy(logits, labels)
            else:
                loss = loss_fn(logits, labels)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

        scaler.step(optimizer)
        scaler.update()

        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()

    return total_loss / max(1, len(data_loader))


Função de Avaliação

In [ ]:
from sklearn.metrics import confusion_matrix

def evaluate_model(model, data_loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch["labels"]

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )

            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision_w = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    recall_w = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
    f1_w = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

    precision_m = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall_m = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    f1_m = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    report = classification_report(
        all_labels,
        all_preds,
        target_names=label_encoder.classes_,
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_weighted": precision_w,
        "recall_weighted": recall_w,
        "f1_weighted": f1_w,
        "precision_macro": precision_m,
        "recall_macro": recall_m,
        "f1_macro": f1_m,
        "report": report,
        "y_true": np.array(all_labels),
        "y_pred": np.array(all_preds),
    }

Validação Cruzada

In [ ]:
from transformers import get_linear_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)

num_epochs = 2
batch_size = 8
n_splits = 10

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

resultados_folds = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f"\n======================")
    print(f"FOLD {fold + 1}/{n_splits}")
    print("======================")

    X_train, y_train = X[train_idx], y[train_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    train_dataset = TextDataset(X_train, y_train)
    test_dataset = TextDataset(X_test, y_test)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_classes
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=1e-5)

    # Pesos de classe
    class_counts = np.bincount(y_train, minlength=num_classes).astype(np.float32)
    class_weights = (class_counts.sum() / (class_counts + 1e-6))
    class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    # Scheduler
    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    for epoch in range(num_epochs):
        train_loss = train_model(
            model=model,
            data_loader=train_loader,
            optimizer=optimizer,
            device=device,
            loss_fn=loss_fn,
            scheduler=scheduler,
            use_amp=True,
            max_grad_norm=1.0
        )
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {train_loss:.4f}")

    eval_out = evaluate_model(model, test_loader, device)

    print("\nRelatório de Classificação:\n", eval_out["report"])
    print(
        f"Acc: {eval_out['accuracy']:.4f} | "
        f"F1(w): {eval_out['f1_weighted']:.4f} | "
        f"F1(macro): {eval_out['f1_macro']:.4f}"
    )

    resultados_folds.append({
        "fold": fold + 1,
        "accuracy": eval_out["accuracy"],
        "precision_weighted": eval_out["precision_weighted"],
        "recall_weighted": eval_out["recall_weighted"],
        "f1_weighted": eval_out["f1_weighted"],
        "precision_macro": eval_out["precision_macro"],
        "recall_macro": eval_out["recall_macro"],
        "f1_macro": eval_out["f1_macro"],
        "report": eval_out["report"]
    })


Usando dispositivo: cuda

FOLD 1/10


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/2 | Loss: 0.3733
Epoch 2/2 | Loss: 0.2698

Relatório de Classificação:
               precision    recall  f1-score   support

         NÃO       0.87      0.84      0.86       715
         SIM       0.93      0.94      0.93      1502

    accuracy                           0.91      2217
   macro avg       0.90      0.89      0.90      2217
weighted avg       0.91      0.91      0.91      2217

Acc: 0.9098 | F1(w): 0.9093 | F1(macro): 0.8957

FOLD 2/10


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/2 | Loss: 0.3754
Epoch 2/2 | Loss: 0.2619

Relatório de Classificação:
               precision    recall  f1-score   support

         NÃO       0.88      0.86      0.87       715
         SIM       0.93      0.95      0.94      1502

    accuracy                           0.92      2217
   macro avg       0.91      0.90      0.90      2217
weighted avg       0.92      0.92      0.92      2217

Acc: 0.9175 | F1(w): 0.9171 | F1(macro): 0.9047

FOLD 3/10


...

Epoch 1/2 | Loss: 0.3794
Epoch 2/2 | Loss: 0.2658

Relatório de Classificação:
               precision    recall  f1-score   support

         NÃO       0.87      0.82      0.84       715
         SIM       0.92      0.94      0.93      1501

    accuracy                           0.90      2216
   macro avg       0.89      0.88      0.89      2216
weighted avg       0.90      0.90      0.90      2216

Acc: 0.9012 | F1(w): 0.9003 | F1(macro): 0.8851

FOLD 9/10


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/2 | Loss: 0.3752
Epoch 2/2 | Loss: 0.2640

Relatório de Classificação:
               precision    recall  f1-score   support

         NÃO       0.86      0.82      0.84       715
         SIM       0.92      0.94      0.93      1501

    accuracy                           0.90      2216
   macro avg       0.89      0.88      0.88      2216
weighted avg       0.90      0.90      0.90      2216

Acc: 0.8994 | F1(w): 0.8988 | F1(macro): 0.8835

FOLD 10/10


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/2 | Loss: 0.3739
Epoch 2/2 | Loss: 0.2609

Relatório de Classificação:
               precision    recall  f1-score   support

         NÃO       0.88      0.87      0.87       715
         SIM       0.94      0.94      0.94      1501

    accuracy                           0.92      2216
   macro avg       0.91      0.91      0.91      2216
weighted avg       0.92      0.92      0.92      2216

Acc: 0.9183 | F1(w): 0.9182 | F1(macro): 0.9063


Resultados

In [ ]:
resultados_df = pd.DataFrame(resultados_folds)

metricas = [
    "accuracy",
    "precision_weighted", "recall_weighted", "f1_weighted",
    "precision_macro", "recall_macro", "f1_macro",
]

print("\n======================")
print("RESUMO CROSS-VALIDATION")
print("======================")

for m in metricas:
    media = resultados_df[m].mean()
    desvio = resultados_df[m].std(ddof=1)
    print(f"{m}: {media:.4f} ± {desvio:.4f}")

# salvar no Drive
saida_csv = "/content/drive/MyDrive/Linguamática/resultados_cv_bert.csv"
resultados_df.drop(columns=["report"]).to_csv(saida_csv, index=False)
print("\nResultados por fold salvos em:", saida_csv)



RESUMO CROSS-VALIDATION
accuracy: 0.9081 ± 0.0062
precision_weighted: 0.9075 ± 0.0064
recall_weighted: 0.9081 ± 0.0062
f1_weighted: 0.9076 ± 0.0064
precision_macro: 0.8983 ± 0.0066
recall_macro: 0.8897 ± 0.0089
f1_macro: 0.8937 ± 0.0075

Resultados por fold salvos em: /content/drive/MyDrive/Linguamática/resultados_cv_bert.csv


# Treino Final

In [ ]:
from transformers import get_linear_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# dataset completo
full_dataset = TextDataset(X, y)
full_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

model_final = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes
).to(device)

optimizer = AdamW(model_final.parameters(), lr=1e-5)

# pesos de classe no dataset inteiro
class_counts = np.bincount(y, minlength=num_classes).astype(np.float32)
class_weights = (class_counts.sum() / (class_counts + 1e-6))
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

total_steps = len(full_loader) * num_epochs
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

for epoch in range(num_epochs):
    train_loss = train_model(
        model=model_final,
        data_loader=full_loader,
        optimizer=optimizer,
        device=device,
        loss_fn=loss_fn,
        scheduler=scheduler,
        use_amp=True,
        max_grad_norm=1.0
    )
    print(f"[FINAL] Epoch {epoch+1}/{num_epochs} | Loss: {train_loss:.4f}")

# salvar modelo e label encoder
output_dir = "/content/drive/MyDrive/Linguamática/modelo_final_bert"
model_final.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

import joblib
joblib.dump(label_encoder, f"{output_dir}/label_encoder.joblib")

print("Modelo final salvo em:", output_dir)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[FINAL] Epoch 1/2 | Loss: 0.3742
[FINAL] Epoch 2/2 | Loss: 0.2705
Modelo final salvo em: /content/drive/MyDrive/Linguamática/modelo_final_bert


Inferência

In [ ]:
import torch
import joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_dir = "/content/drive/MyDrive/Linguamática/modelo_final_bert"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
label_encoder = joblib.load(f"{model_dir}/label_encoder.joblib")

def predict(texts, max_length=128):
    model.eval()
    if isinstance(texts, str):
        texts = [texts]

    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = model(**enc).logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
    return label_encoder.inverse_transform(preds)

# exemplo
print(predict(["eu amo essa música", "essa mulher é muito feia"]))

['NÃO' 'SIM']
